# 15 — Experiment 2: synthetic historical control (own history as donor and as fill)

Donors = the treated site's **own pre-treatment periods**, arranged as historical blocks
(Chen, Yang & Yang, *Synthetic Historical Control*, SSRN 4995085): the treated block is the last
m training periods plus the target; historical block i is the same (m pre, 1 post) window shifted
back i periods, all inside the training window (N = 8 − m blocks for P09, 9 − m for P10).
Simplex weights are fitted on the pre segments and applied to the blocks' post periods. No
alignment: same parcel, same class, same position. Cloud handling = the notebook-14 historical
fill (`latents_biweekly_histfill.npz`), with the chip-mean cache alongside for comparison.

Scorer = notebook 11 (pooled per-dimension scaler over all 60 sites' training periods, SLSQP
simplex, missing rows dropped, per-site standardized test RMSE with train alongside, mean over
10 sites, S1 before S2). Representations: 980-d latent, chip mean (5), Gram (15).

| arm | unit | donor pool |
|---|---|---|
| 2-H0 | site | own-history mean over the training periods, single donor (= C2 benchmark) |
| 2-L | site | last training period (persistence) |
| 2-H1 | site | own SHC blocks, m ∈ {2, 3, 4} |
| 2-P1 | parcel | per position: own SHC blocks of that parcel (5 ch × m rows), m = 3 |
| 2-X | site | hybrid: own SHC blocks (m = 3) + the 5 Experiment-1 aligned donors (arm D_min), one simplex on the same m periods; donors-only on those m periods as its control |

Each arm is run on levels and on first differences of the standardized series (SSRN: detrend a
trending series); differenced predictions are added back to the last observed level so all
RMSEs are in levels. C1 = test ≤ 1.5 × train; C2 = beats 2-H0; C3 (2-X only) = beats equal-weight.


In [1]:
import sys, itertools
import numpy as np, pandas as pd
sys.path.insert(0, ".")
import panel_lib as pl, panel_repr as pr, panel_align as pa
pd.set_option("display.width", 220)

CACHES = {"chipmean": "latents_biweekly.npz", "histfill": "latents_biweekly_histfill.npz"}
PANELS = {c: pl.Panel.from_npz(pl.LATD / n) for c, n in CACHES.items()}
ALL_SITES = sorted(PANELS["chipmean"].roster["site_id"]); TREAT = PANELS["chipmean"].treatments
DON = pr.matched_donors(PANELS["chipmean"])
_z = np.load("parcel_perms.npz"); PERMS = {tuple(k.split("|")): _z[k] for k in _z.files}
TESTS = ((9, range(1, 9)), (10, range(1, 10)))
REPRS = ["latent980", "chip_mean", "gram"]


def feat(panel, sensor, name, s, q):
    v = panel.L(s, sensor, q)
    if v is None: return np.full(980 if name == "latent980" else pr.dim_of(name), np.nan)
    return v if name == "latent980" else pr._repr_raw(v.reshape(5, 196), name)


def zfun(panel, sensor, name, train_p):
    T = np.stack([feat(panel, sensor, name, s, q) for s in ALL_SITES for q in train_p])
    mu, sd = np.nanmean(T, 0), np.nanstd(T, 0, ddof=1); sd[~np.isfinite(sd) | (sd == 0)] = 1.0
    return lambda v: (v - mu) / sd


def series(panel, sensor, name, s, z, diff=False):
    """{seq: standardized vector (levels) or first difference z[q]-z[q-1]} for P01-P10."""
    L = {q: z(feat(panel, sensor, name, s, q)) for q in range(1, 11)}
    if not diff: return L
    return {q: (L[q] - L[q - 1]) if q > 1 else np.full_like(L[q], np.nan) for q in range(1, 11)}


def shc_fit(Y, train_p, test_q, m, min_rows=1):
    """Y: {seq: vector}. Returns (pred, train_rmse, n_blocks, w) — site-level SHC."""
    pre_t = pa.treated_block(train_p, m)
    y = np.concatenate([Y[q] for q in pre_t])
    cols, posts = [], []
    for pre, post in pa.shc_blocks(train_p, m):
        x = np.concatenate([Y[q] for q in pre]); xp = Y[post[0]]
        if np.isfinite(x).any() and np.isfinite(xp).any():
            cols.append(x); posts.append(xp)
    if len(cols) < 1: return None, np.nan, 0, None
    X = np.column_stack(cols)
    if not (np.isfinite(y) & np.isfinite(X).all(1)).any(): return None, np.nan, 0, None
    w = pa.simplex_scm(y, X)
    return np.column_stack(posts) @ w, pa.rmse(y - X @ w), len(cols), w
print("ready")


ready


## 1. Site-level arms — H0, L, H1 (m = 2, 3, 4); levels and first differences; both caches

In [2]:
def run_site(cache, sensor, name):
    panel = PANELS[cache]; rows = []
    for test_q, train_p in TESTS:
        z = zfun(panel, sensor, name, train_p)
        for t in TREAT:
            L = series(panel, sensor, name, t, z); D = series(panel, sensor, name, t, z, diff=True)
            ytrue = L[test_q]
            h0 = np.nanmean(np.stack([L[q] for q in train_p]), 0)
            base = {"cache": cache, "sensor": sensor, "repr": name, "site": t, "test": f"P{test_q:02d}",
                    "own_hist_rmse": pa.rmse(ytrue - h0)}
            rows.append({**base, "arm": "H0_own_mean", "test_rmse": pa.rmse(ytrue - h0), "train_rmse": np.nan, "n_blocks": 0})
            rows.append({**base, "arm": "L_last_period", "test_rmse": pa.rmse(ytrue - L[test_q - 1]), "train_rmse": np.nan, "n_blocks": 0})
            for m in (2, 3, 4):
                p, tr, nb_, _ = shc_fit(L, train_p, test_q, m)
                rows.append({**base, "arm": f"H1_m{m}", "test_rmse": pa.rmse(ytrue - p) if p is not None else np.nan, "train_rmse": tr, "n_blocks": nb_})
                p, tr, nb_, _ = shc_fit(D, train_p, test_q, m)
                pred = (L[test_q - 1] + p) if p is not None else None
                rows.append({**base, "arm": f"H1_m{m}_diff", "test_rmse": pa.rmse(ytrue - pred) if pred is not None else np.nan, "train_rmse": tr, "n_blocks": nb_})
    df = pd.DataFrame(rows)
    df["C1"] = df.test_rmse <= 1.5 * df.train_rmse; df["C2"] = df.test_rmse < df.own_hist_rmse
    return df

SITE = pd.concat([run_site(c, s, n) for c in CACHES for s in pl.SENSORS for n in REPRS], ignore_index=True)

def summ(df, extra=()):
    g = df.groupby(["sensor", "cache", "repr", "arm", "test"])
    out = g[["test_rmse", "train_rmse"]].mean().round(3)
    out["C1"] = g.C1.sum(min_count=1); out["C2"] = g.C2.sum(min_count=1); out["n"] = g.test_rmse.count()
    for e in extra: out[e] = g[e].sum(min_count=1)
    return out

ARM_ORDER = ["H0_own_mean", "L_last_period", "H1_m2", "H1_m3", "H1_m4", "H1_m2_diff", "H1_m3_diff", "H1_m4_diff"]
for sensor in pl.SENSORS:
    for name in REPRS:
        print(f"\n=== {sensor} — {name} ===")
        s = summ(SITE.query("sensor == @sensor and repr == @name")).reset_index()
        piv = s.pivot_table(index=["arm"], columns=["cache", "test"], values=["test_rmse", "train_rmse", "C2"]).reindex(ARM_ORDER)
        print(piv.round(3).to_string())



=== sentinel1 — latent980 ===
                    C2                    test_rmse                        train_rmse                       
cache         chipmean      histfill       chipmean        histfill          chipmean        histfill       
test               P09  P10      P09  P10       P09    P10      P09    P10        P09    P10      P09    P10
arm                                                                                                         
H0_own_mean        0.0  0.0      0.0  0.0     0.689  0.690    0.689  0.690        NaN    NaN      NaN    NaN
L_last_period      0.0  0.0      0.0  0.0     0.879  0.911    0.879  0.911        NaN    NaN      NaN    NaN
H1_m2              0.0  0.0      0.0  0.0     0.709  0.701    0.709  0.701      0.691  0.687    0.691  0.687
H1_m3              0.0  0.0      0.0  0.0     0.721  0.704    0.721  0.704      0.731  0.695    0.731  0.695
H1_m4              0.0  0.0      0.0  0.0     0.734  0.715    0.734  0.715      0.736  0.728    0

### Reference: cross-sectional numbers in the same metric (notebook 11 / Experiment 1)

In [3]:
ref = pd.read_csv("panel_collabstyle_scm_validation.csv").query("cache == 'chipmean' and repr in @REPRS")
print("notebook 11, collaborator's 5 donors, chip-mean cache (mean test RMSE (train)):")
print(ref.pivot_table(index=["sensor", "repr"], columns="test", values=["mean_test_rmse", "mean_train_rmse"]).round(3).to_string())
e1 = pd.read_csv("panel_align_exp1_validation.csv").query("arm in ['A', 'D_min', 'G_block2_D_min']")
print("\nExperiment 1 (latent980, per-channel scaler):")
print(e1.pivot_table(index=["sensor", "arm"], columns="test", values=["test_rmse", "train_rmse"]).round(3).to_string())


notebook 11, collaborator's 5 donors, chip-mean cache (mean test RMSE (train)):
                    mean_test_rmse        mean_train_rmse       
test                           P09    P10             P09    P10
sensor    repr                                                  
sentinel1 chip_mean          0.603  0.701           0.587  0.596
          gram               0.692  0.732           0.659  0.665
          latent980          1.084  1.087           1.093  1.093
sentinel2 chip_mean          0.577  0.395           0.741  0.736
          gram               0.625  0.466           0.734  0.731
          latent980          1.107  1.251           0.954  0.960

Experiment 1 (latent980, per-channel scaler):
                         test_rmse        train_rmse       
test                           P09    P10        P09    P10
sensor    arm                                              
sentinel1 A                  1.079  1.082      1.088  1.088
          D_min              0.791  0.797      0

## 2. Parcel-level 2-P1 — each position fitted on its own history blocks (m = 3), 980-d

In [4]:
def run_parcel(cache, sensor, m=3, diff=False):
    panel = PANELS[cache]; rows = []
    for test_q, train_p in TESTS:
        z = zfun(panel, sensor, "latent980", train_p)
        for t in TREAT:
            Lv = series(panel, sensor, "latent980", t, z); Dv = series(panel, sensor, "latent980", t, z, diff=True)
            Y = Dv if diff else Lv
            R = {q: Y[q].reshape(5, 196) for q in Y}
            pred = np.full((5, 196), np.nan); e_tr = []
            for p in range(196):
                Yp = {q: R[q][:, p] for q in R}
                pr_, tr, nb_, w = shc_fit(Yp, train_p, test_q, m)
                if pr_ is None: continue
                pred[:, p] = pr_
                pre_t = pa.treated_block(train_p, m)
                y = np.concatenate([Yp[q] for q in pre_t])
                X = np.column_stack([np.concatenate([Yp[q] for q in pre]) for pre, post in pa.shc_blocks(train_p, m)
                                     if np.isfinite(np.concatenate([Yp[q] for q in pre])).any() and np.isfinite(Yp[post[0]]).any()])
                e_tr.append(y - X @ w)
            ytrue = Lv[test_q].reshape(5, 196)
            if diff: pred = Lv[test_q - 1].reshape(5, 196) + pred
            h0 = np.nanmean(np.stack([Lv[q] for q in train_p]), 0).reshape(5, 196)
            te = pa.rmse((ytrue - pred).ravel()); tr = pa.rmse(np.concatenate(e_tr)) if e_tr else np.nan
            rows.append({"cache": cache, "sensor": sensor, "repr": "latent980", "site": t, "test": f"P{test_q:02d}",
                         "arm": f"P1_m{m}" + ("_diff" if diff else ""), "test_rmse": te, "train_rmse": tr,
                         "own_hist_rmse": pa.rmse((ytrue - h0).ravel()), "n_blocks": 8 - m if test_q == 9 else 9 - m})
    df = pd.DataFrame(rows); df["C1"] = df.test_rmse <= 1.5 * df.train_rmse; df["C2"] = df.test_rmse < df.own_hist_rmse
    return df

P1 = pd.concat([run_parcel(c, s, 3, d) for c in CACHES for s in pl.SENSORS for d in (False, True)], ignore_index=True)
print(summ(P1).reset_index().pivot_table(index=["sensor", "arm"], columns=["cache", "test"], values=["test_rmse", "train_rmse", "C2"]).round(3).to_string())
SITE = pd.concat([SITE, P1], ignore_index=True)


                           C2                    test_rmse                        train_rmse                       
cache                chipmean      histfill       chipmean        histfill          chipmean        histfill       
test                      P09  P10      P09  P10       P09    P10      P09    P10        P09    P10      P09    P10
sensor    arm                                                                                                      
sentinel1 P1_m3           0.0  0.0      0.0  0.0     0.795  0.777    0.795  0.777      0.624  0.574    0.624  0.574
          P1_m3_diff      0.0  0.0      0.0  0.0     0.998  1.003    0.998  1.003      0.765  0.685    0.765  0.685
sentinel2 P1_m3           1.0  3.0      0.0  0.0     1.119  1.061    0.961  0.984      0.865  0.782    0.554  0.659
          P1_m3_diff      0.0  0.0      0.0  0.0     1.335  1.325    1.133  1.182      1.045  0.892    0.701  0.824


## 3. Hybrid 2-X — own SHC blocks (m = 3) + Experiment-1 aligned donors in one simplex (980-d, per-channel scaler)

Rows = the treated block's m pre periods; cross-sectional donor columns = aligned donor at those periods, predicting with the donor at the target; block columns = own history, predicting with the block's post period. `X_donors_m3` = the same 5 aligned donors fitted on those m periods only (control).

In [5]:
def scaler_pc(panel, sensor, train_p):
    T = np.stack([panel.L(s, sensor, q) for s in ALL_SITES for q in train_p if panel.L(s, sensor, q) is not None])
    V = T.reshape(len(T), 5, 196).transpose(1, 0, 2).reshape(5, -1)
    mu, sd = np.repeat(V.mean(1), 196), np.repeat(V.std(1, ddof=1), 196); sd[sd == 0] = 1.0
    return lambda v: (v - mu) / sd

def run_hybrid(cache, sensor, m=3):
    panel = PANELS[cache]; rows = []
    for test_q, train_p in TESTS:
        z = scaler_pc(panel, sensor, train_p)
        for t in TREAT:
            L = {q: z(feat(panel, sensor, "latent980", t, q)) for q in range(1, 11)}
            pre_t = pa.treated_block(train_p, m)
            y = np.concatenate([L[q] for q in pre_t]); ytrue = L[test_q]
            dcols, dte = [], []
            for j in DON[t]:
                perm = PERMS[("D_min", sensor, t, j)]
                if (perm >= 0).sum() < pa.MIN_MATCHED: continue
                dcols.append(np.concatenate([pa.aligned_vec(panel, j, sensor, q, perm) for q in pre_t]))
                dte.append(pa.aligned_vec(panel, j, sensor, test_q, perm))
            bcols, bte = [], []
            for pre, post in pa.shc_blocks(train_p, m):
                x = np.concatenate([L[q] for q in pre]); xp = L[post[0]]
                if np.isfinite(x).any() and np.isfinite(xp).any(): bcols.append(x); bte.append(xp)
            h0 = np.nanmean(np.stack([L[q] for q in train_p]), 0)
            for arm, cols, tes in (("X_donors_m3", dcols, dte), ("X_hybrid_m3", dcols + bcols, dte + bte), ("X_blocks_m3", bcols, bte)):
                X = np.column_stack(cols); w = pa.simplex_scm(y, X); Xt = np.column_stack(tes)
                te = pa.rmse(ytrue - Xt @ w)
                rows.append({"cache": cache, "sensor": sensor, "repr": "latent980", "site": t, "test": f"P{test_q:02d}", "arm": arm,
                             "test_rmse": te, "train_rmse": pa.rmse(y - X @ w), "own_hist_rmse": pa.rmse(ytrue - h0),
                             "n_blocks": len(bcols) if "donors" not in arm else 0, "n_donors": len(dcols) if "blocks" not in arm else 0,
                             "w_blocks_total": float(w[len(dcols):].sum()) if arm == "X_hybrid_m3" else np.nan,
                             "C3": te < pa.rmse(ytrue - np.nanmean(Xt, 1))})
    df = pd.DataFrame(rows); df["C1"] = df.test_rmse <= 1.5 * df.train_rmse; df["C2"] = df.test_rmse < df.own_hist_rmse
    return df

X = pd.concat([run_hybrid(c, s) for c in CACHES for s in pl.SENSORS], ignore_index=True)
print(summ(X, extra=("C3",)).reset_index().pivot_table(index=["sensor", "arm"], columns=["cache", "test"], values=["test_rmse", "train_rmse", "C2", "C3"]).round(3).to_string())
print("\nshare of simplex weight on own-history blocks in the hybrid (mean over sites):")
print(X.query("arm == 'X_hybrid_m3'").groupby(["sensor", "cache", "test"]).w_blocks_total.mean().round(2).to_string())
SITE = pd.concat([SITE, X], ignore_index=True)


/tmp/ipykernel_1826966/183025073.py:33: RuntimeWarning: Mean of empty slice
  "C3": te < pa.rmse(ytrue - np.nanmean(Xt, 1))})


/tmp/ipykernel_1826966/183025073.py:33: RuntimeWarning: Mean of empty slice
  "C3": te < pa.rmse(ytrue - np.nanmean(Xt, 1))})


/tmp/ipykernel_1826966/183025073.py:33: RuntimeWarning: Mean of empty slice
  "C3": te < pa.rmse(ytrue - np.nanmean(Xt, 1))})


/tmp/ipykernel_1826966/183025073.py:33: RuntimeWarning: Mean of empty slice
  "C3": te < pa.rmse(ytrue - np.nanmean(Xt, 1))})


                            C2                          C3                    test_rmse                        train_rmse                       
cache                 chipmean      histfill      chipmean      histfill       chipmean        histfill          chipmean        histfill       
test                       P09  P10      P09  P10      P09  P10      P09  P10       P09    P10      P09    P10        P09    P10      P09    P10
sensor    arm                                                                                                                                   
sentinel1 X_blocks_m3      0.0  0.0      0.0  0.0      7.0  3.0      7.0  3.0     0.716  0.699    0.716  0.699      0.726  0.690    0.726  0.690
          X_donors_m3      0.0  0.0      0.0  0.0      7.0  7.0      7.0  7.0     1.005  0.990    1.005  0.990      0.983  0.984    0.983  0.984
          X_hybrid_m3      3.0  4.0      3.0  4.0      7.0  6.0      7.0  6.0     0.722  0.704    0.722  0.704      0.721  0.691  

## 4. Cache comparison under a common scaler

Standardized RMSEs are in units of the pooled training-period SD of each cache. Replacing fully masked chips (encoded as constant images under chip-mean fill) by history templates shrinks that SD, so the same absolute error looks larger under the histfill scaler. This cell scores both caches with the **chip-mean cache's scaler** so the fill's effect is isolated from the yardstick.

In [6]:
rows = []
for sensor in ["sentinel2"]:
    for name in REPRS:
        for test_q, train_p in TESTS:
            z_cm = zfun(PANELS["chipmean"], sensor, name, train_p)
            z_hf = zfun(PANELS["histfill"], sensor, name, train_p)
            T_cm = np.stack([feat(PANELS["chipmean"], sensor, name, s, q) for s in ALL_SITES for q in train_p])
            T_hf = np.stack([feat(PANELS["histfill"], sensor, name, s, q) for s in ALL_SITES for q in train_p])
            sd_ratio = float(np.nanmedian(np.nanstd(T_hf, 0, ddof=1) / np.nanstd(T_cm, 0, ddof=1)))
            for cache in CACHES:
                panel = PANELS[cache]
                for t in TREAT:
                    L = series(panel, sensor, name, t, z_cm)
                    ytrue = L[test_q]; h0 = np.nanmean(np.stack([L[q] for q in train_p]), 0)
                    p3, tr3, _, _ = shc_fit(L, train_p, test_q, 3)
                    rows.append({"sensor": sensor, "repr": name, "cache": cache, "test": f"P{test_q:02d}", "site": t,
                                 "sd_ratio_hf_over_cm": sd_ratio, "H0_own_mean": pa.rmse(ytrue - h0),
                                 "H1_m3": pa.rmse(ytrue - p3) if p3 is not None else np.nan})
CS = pd.DataFrame(rows)
print("Sentinel-2, both caches scored with the chip-mean cache scaler (mean over 10 sites); "
      "sd_ratio = median over dims of histfill SD / chip-mean SD:")
print(CS.groupby(["repr", "cache", "test"])[["H0_own_mean", "H1_m3", "sd_ratio_hf_over_cm"]].mean().round(3).to_string())
CS.to_csv("panel_hist_exp2_commonscaler.csv", index=False)


Sentinel-2, both caches scored with the chip-mean cache scaler (mean over 10 sites); sd_ratio = median over dims of histfill SD / chip-mean SD:
                         H0_own_mean  H1_m3  sd_ratio_hf_over_cm
repr      cache    test                                         
chip_mean chipmean P09         0.807  1.064                0.594
                   P10         0.790  0.917                0.606
          histfill P09         0.358  0.443                0.594
                   P10         0.772  0.600                0.606
gram      chipmean P09         0.868  1.154                0.514
                   P10         0.983  1.131                0.523
          histfill P09         0.390  0.455                0.514
                   P10         0.811  0.658                0.523
latent980 chipmean P09         0.992  1.070                1.079
                   P10         0.989  0.996                1.072
          histfill P09         0.851  0.861                1.079
           

In [7]:
SITE.to_csv("panel_hist_exp2_sites.csv", index=False)
MEAN = summ(SITE, extra=("C3",)).reset_index(); MEAN.to_csv("panel_hist_exp2_validation.csv", index=False)
print("saved panel_hist_exp2_validation.csv / panel_hist_exp2_sites.csv")
print("\nheadline (latent980): mean standardized test RMSE (train)")
print(MEAN.query("repr == 'latent980'").pivot_table(index=["sensor", "arm"], columns=["cache", "test"], values=["test_rmse", "train_rmse"]).round(3).to_string())


saved panel_hist_exp2_validation.csv / panel_hist_exp2_sites.csv

headline (latent980): mean standardized test RMSE (train)
                        test_rmse                        train_rmse                       
cache                    chipmean        histfill          chipmean        histfill       
test                          P09    P10      P09    P10        P09    P10      P09    P10
sensor    arm                                                                             
sentinel1 H0_own_mean       0.689  0.690    0.689  0.690        NaN    NaN      NaN    NaN
          H1_m2             0.709  0.701    0.709  0.701      0.691  0.687    0.691  0.687
          H1_m2_diff        0.909  0.925    0.909  0.925      0.897  0.890    0.897  0.890
          H1_m3             0.721  0.704    0.721  0.704      0.731  0.695    0.731  0.695
          H1_m3_diff        0.914  0.927    0.914  0.927      0.944  0.900    0.944  0.900
          H1_m4             0.734  0.715    0.734  0.715 

## Reading

1. **Sentinel-1 (cache-invariant): own history is a better 980-d donor than the aligned
   cross-sectional donors, and the mean is all that matters.** Own-history mean (H0)
   0.689 / 0.690 vs Experiment-1 arm D_min 0.791 / 0.797 and same-coordinate 1.08. SHC weights
   do not improve on the mean (H1: 0.70–0.73 for m = 2, 3, 4; train ≈ test), differencing hurts
   (0.91–0.94), persistence (last period) is worse (0.879 / 0.911), and per-parcel weights (P1)
   are worse (0.795 / 0.777) with train 0.57–0.62 — 5 blocks × 3 periods overfit a 15-row fit.
   In the hybrid the simplex puts 84–88 % of the weight on own blocks and lands at 0.722 / 0.704;
   the 5 aligned donors fitted on the same 3 periods alone give 1.005 / 0.990. The best
   Sentinel-1 number remains Experiment-1 arm G (2×2 block means of aligned donors, 0.428 / 0.440).
2. **Sentinel-1 pooled reps: history ≈ cross-sectional at P09, better at P10.** Chip mean: H0
   0.604 vs notebook-11 SCM 0.603 at P09; at P10 persistence 0.531 and H1_m3 0.514 beat SCM's
   0.701. Gram: H1_m4 0.510 vs SCM 0.732 at P10. SHC train RMSEs (0.42–0.57) are well below test
   at P09 for chip mean (0.73–0.84): a 5-d fit on 3–4 periods is not constrained.
3. **Sentinel-2, 980-d: historical fill + own history is the best spatial result so far.** With
   the histfill cache, H0 0.864 / 0.925 and H1_m3 0.875 / 0.899 (C2 5/10 and 7/10) beat arm D_min
   (0.992 / 0.903); the hybrid reaches **0.786 / 0.866** (C2 8/10 both tests) with 86–89 % of the
   weight on own blocks. Under the chip-mean cache the same arms sit at 0.99–1.07.
4. **Scaler caveat for cross-cache comparisons (section 4).** The pooled SD of chip-mean and Gram
   features shrinks to 0.51–0.61 of its chip-mean-cache value once the 145 fully masked chips
   (constant images under chip-mean fill) are replaced by history templates, so histfill numbers
   in their own scaler look worse for pooled reps (chip mean H0 at P10: 1.297 vs 0.790). Scored
   under the common chip-mean scaler the fill helps everywhere: chip mean H0 0.358 / 0.772 vs
   0.807 / 0.790, H1_m3 0.443 / 0.600 vs 1.064 / 0.917; Gram H1_m3 0.455 / 0.658 vs 1.154 / 1.131;
   980-d H0 0.851 / 0.921 vs 0.992 / 0.989. The 980-d scaler is unaffected (ratio 1.07–1.08).
5. **What the P09 vs P10 split means.** At P09, 8 of the 10 treated targets are themselves
   history-filled (site 0010: 94 % of pixels), so part of the P09 gain is the target resembling
   the template the predictor is built from. **P10 is the clean-target test** (only site 0004's
   permanent edge column is filled): there the fill still cuts the SHC error by a third for pooled
   reps (0.917 → 0.600 chip mean, 1.131 → 0.658 Gram) and by 10 % for 980-d, because the
   *training* chips are cleaner. That is the defensible claim.
6. **Cross-sectional donors still win for Sentinel-2 pooled reps on the clean target**: notebook-11
   SCM chip mean 0.395 at P10 vs the best historical arm 0.600 (common scaler). Own history
   replaces donors only in the spatial (980-d) representation, where donors were never usable
   without alignment.
